# ~ TilStat scraper ~

Utility to scrape the scores to enter the various courses over the years. Like any scrape utility, this code sucks.

In [21]:
import requests
from bs4 import BeautifulSoup as bs
from multiprocessing import Pool
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.notebook import trange, tqdm

## Graduatorie 21-22

<b>Info importante:</b> la convenzione Politecnica è di riferirsi all'anno accademico con <b>l'ultimo anno del biennio</b>. Quindi il 2021/22 è definito come 2022.

Da quest'anno il formato delle statistiche è stato aggiornato. I livelli ora sono:

1.) Elenco generale, es: http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing_sep?p_a_acc=2022 <br>
1.1) Elenco per corso, es: http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing_sep?p_a_acc=2022&p_mnemo=ECE1T3 <br>
1.1.1) Singola graduatoria, es: http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing_sep?p_data=07/09/2021&p_a_acc=2022&p_mnemo=ECE1T3
 

In [9]:
base = 'http://didattica.polito.it'

In [14]:
indexes = []

# legge gli elenchi generali (1.) di normali e contingentati
# il loop è per generalizzare, ad oggi questo format è solo per l'anno 2022
for year in [2022]:
    indexes += [
        # normali
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.ing_sep?p_a_acc={year}',
         'conting': False, 'course': 'Ingegneria', 'year':year},
        
        # contingentati
        {'uri':f'{base}/portal30/preimma.pkg_grad.ing_sep_cont?p_a_acc={year}',
         'conting': True, 'course': 'Ingegneria', 'year':year},
    ]
    
# partendo dagli elenchi generali, legge gli elenchi per corso
explode = []
for index in indexes:
    html = requests.get(index['uri']).text
    soup = bs(html, 'html.parser')
    
    subindex = [link for link in soup.find_all("a")
                     if 'p_mnemo=' in link['href'] and "L-" in link.text]
    
    for s in subindex:
        explode.append({
            **index,
            'corso unico': s.text.split('(')[0].strip(),
            'uri': base+s['href']
        })
        
# per analogia con gli anni precedenti, rinomino la variabile
indexes = explode
assert len(indexes) > 0

In [23]:
# memorizzo i link di ciascuna graduatoria (rank)

ranks = []
for index in tqdm(indexes):
    html = requests.get(index['uri']).text
    soup = bs(html, 'html.parser')
    
    ranklinks = [link for link in soup.find_all("a")
                     if 'preimma' in link['href'] and link.text.startswith('Graduatoria del ')]
    for i, link in enumerate(ranklinks):
        
        # paranoid check
        # Es: Ingegneria has .ing string in each rank
        assert f".{index['course'].lower()[:3]}" in link['href'], "Wrong link assoc..."
        
        date = [int(x) for x in link.text.replace("Graduatoria del ","").split('/')]
        date = datetime(date[2],date[1],date[0])
        
        # paranoid check
        first_sane_rank = datetime(int(index['year'])-1, 7, 1) # dal primo luglio del primo anno di inizio coorte
        last_sane_rank = datetime(int(index['year']), 3, 15) # a metà marzo del secondo anno di inizio coorte
        assert first_sane_rank < date < last_sane_rank, "Wrong year assoc..."
        
        ranks.append({
            'uri': base+link['href'],
            'conting': index['conting'],
            'course': index['course'],
            'corso unico': index['corso unico'],
            'year': index['year'],
            'grad_n': i,
            'date': date
        })

  0%|          | 0/21 [00:00<?, ?it/s]

In [24]:
# printa alcune cose, per controllare che sia tutto ok prima del task lungo
print(ranks[0])
print(len(ranks))

{'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing_sep?p_data=07/09/2021&p_a_acc=2022&p_mnemo=ECE1T3', 'conting': False, 'course': 'Ingegneria', 'corso unico': 'ELECTRONIC AND COMMUNICATIONS ENGINEERING', 'year': 2022, 'grad_n': 0, 'date': datetime.datetime(2021, 9, 7, 0, 0)}
231


In [27]:
# dataframe che conterrà ogni riga di ogni graduatoria di ogni anno (in questo caso solo il 2022)
dfs = []

# giro sulle graduatorie ed inserisco ogni riga in dfs
# si potrebbe parallellizzare ma tende a bloccarsi, con una discreta connessione ci vanno 5 minuti
for rank in tqdm(ranks):
    
    html = requests.get(rank['uri']).text
    soup = bs(html, 'html.parser')
    
    headers = soup.select("table th")
    rows = soup.select("table tr")
    cells = soup.select("table td")

    # paranoid check
    assert len(headers)*(len(rows)-1) == len(cells), "Wrong table format"

    headervalues = [str(h.text.lower().strip()) for h in headers]
    values = [str(c.text.strip()) for c in cells]
    shapedvalues = np.asarray(values).reshape((len(rows)-1),len(headers)).astype(str)
    df = pd.DataFrame(shapedvalues, columns=headervalues)

    # remove special students
    df = df[df["totale"] != "Certificazione Internazionale"]
    
    df["totale"] = df["totale"].str.replace(',','.').astype(float)
    df['uri'] = rank['uri']
    df['conting'] = rank['conting']
    df['course'] = rank['course']
    df['corso unico'] = rank['corso unico']
    df['grad_n'] = rank['grad_n']
    df['date'] = rank['date']
    df['year'] = rank['year']
    
    dfs.append(df)

  0%|          | 0/231 [00:00<?, ?it/s]

In [28]:
result = pd.concat(dfs, ignore_index=True)

In [30]:
result.to_csv(f"ranks/post2022_{str(datetime.now()).replace(':','-')}.csv")

In [31]:
len(dfs[1])

49

In [32]:
result

,af user,totale,matematica,comp.del testo,logica,fisica,esito,uri,conting,course,corso unico,grad_n,date,year
0,F408172,91.07,"16,75",6,"4,75","10,75",IMMATRICOLATO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,ELECTRONIC AND COMMUNICATIONS ENGINEERING,0,2021-09-07,2022
1,F417123,83.33,"14,75","4,75","4,75","10,75",ASSEGNATO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,ELECTRONIC AND COMMUNICATIONS ENGINEERING,0,2021-09-07,2022
2,F420608,82.14,"16,75","4,75","4,75","8,25",IMMATRICOLATO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,ELECTRONIC AND COMMUNICATIONS ENGINEERING,0,2021-09-07,2022
3,F407012,70.24,"14,5","4,75",4,"6,25",IMMATRICOLATO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,ELECTRONIC AND COMMUNICATIONS ENGINEERING,0,2021-09-07,2022
4,F423629,68.45,"9,25",6,"4,75","8,75",IMMATRICOLATO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,ELECTRONIC AND COMMUNICATIONS ENGINEERING,0,2021-09-07,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86515,F406906,16.67,"2,75","3,75","1,75","-1,25",NON AMMESSO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,MATEMATICA PER L'INGEGNERIA,10,2021-10-29,2022
86516,F408894,16.07,"2,75","2,5","1,75","-0,25",NON AMMESSO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,MATEMATICA PER L'INGEGNERIA,10,2021-10-29,2022
86517,F62089,16.07,"1,75","2,25","0,25","2,5",NON AMMESSO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,MATEMATICA PER L'INGEGNERIA,10,2021-10-29,2022
86518,F436250,14.88,3,"2,5",0,"0,75",NON AMMESSO,http://didattica.polito.it/pls/portal30/preimm...,False,Ingegneria,MATEMATICA PER L'INGEGNERIA,10,2021-10-29,2022
